# 13. Stomach mct mc

Part of the **[Fig. 6 chapter](fig6.md)** — see it for the panel-by-panel map, run order, and data inputs. The first code cell sets `ENTEX_ROOT` and activates the no-overwrite guard (see the [Reproduction guide](../reproduce.md)).


In [ ]:
# === Reproduction setup — edit ENTEX_ROOT / REF_ROOT for your machine ===
# All absolute paths below resolve from these two roots; the working directory
# is the original analysis/ folder, and repro_guard prevents any existing file
# from being overwritten when you re-run this notebook. See the book's
# 'Reproduction guide' chapter for details.
import os, sys
ENTEX_ROOT = os.environ.get('ENTEX_ROOT', '/large_storage/zhoulab/zhoujt/project/ENTEx')
REF_ROOT   = os.environ.get('REF_ROOT',   '/large_storage/zhoulab/ref')
BOOK_ROOT  = os.environ.get('BOOK_ROOT',  f'{ENTEX_ROOT}/analysis/HumanCellEpigenomeAtlas')
sys.path.insert(0, BOOK_ROOT)
os.chdir(f'{ENTEX_ROOT}/analysis')   # original working directory
import repro_guard                    # no-overwrite guard (default: skip existing)


In [ ]:
import os
from glob import glob
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

import cooler
import anndata
import scanpy as sc
import scanpy.external as sce
from ALLCools.clustering import *
from ALLCools.plot import *
from ALLCools.mcds import MCDS
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
# mpl.use('agg')
mpl.style.use("default")
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = "Helvetica"

import warnings
warnings.filterwarnings("ignore")


In [ ]:
def dump_embedding(adata, name, n_dim=2):
    # put manifold coordinates into adata.obs
    for i in range(n_dim):
        adata.obs[f'{name}_{i}'] = adata.obsm[f'X_{name}'][:, i]
    return adata


In [ ]:
indir = f'{ENTEX_ROOT}/'
mct_dir = f'{indir}mCT/'
m3c_dir = f'{indir}mcds/'
outdir = f'{indir}analysis/stomach_mct/'


In [ ]:
var_dim = 'chrom5k'
chrom_to_remove = ['chrX', 'chrY', 'chrM', 'chrL']


In [ ]:
meta = anndata.read_h5ad(f'{mct_dir}entex_1_rna.h5ad', 'r').obs
selc = meta.index[(meta['UniqueAlignFinalReads']>500000) & (meta['mCCCFrac']<0.1)]
print(len(selc))


In [ ]:
mcds = MCDS.open(f'{mct_dir}entex_1.mcds', use_obs=selc, var_dim=var_dim)
mcds = mcds.remove_chromosome(exclude_chromosome=chrom_to_remove, var_dim=var_dim)
mcds

In [ ]:
mcad = mcds.get_score_adata(mc_type='CGN', quant_type='hypo')
binarize_matrix(mcad, cutoff=0.95)


In [ ]:
black_list_path = f'{REF_ROOT}/blacklist/hg38-blacklist.v2.bed.gz'
filter_regions(mcad, n_cell=5)
remove_black_list_region(mcad, black_list_path=black_list_path)


In [ ]:
mcad.write_h5ad(f'{outdir}5kCG.h5ad')

In [ ]:
model = LSI(scale_factor=10000,
            n_components=50,
            algorithm='arpack',
            random_state=0)


In [ ]:
raw_key = '5kCG_lsi'
obsm_key = 'X_lsi'
# adata_mc_list = []
# for xx in adata.obs['Donor'].unique():
    # tmp = mcad[adata.obs['Donor']==xx].copy()
model.fit_transform(mcad, obsm_name='5kCG_lsi')
npc = significant_pc_test(mcad, p_cutoff=0.1, obsm=raw_key, update=False)


In [ ]:
mcad.obs['Donor'] = mcad.obs.index.str.split('_').str[1]
mcad.obs['Donor'].value_counts()

In [ ]:
mcad.obsm[obsm_key] = normalize(mcad.obsm[raw_key][:, :npc], axis=1)
tsne(mcad, obsm=obsm_key, metric='euclidean', exaggeration=-1, perplexity=50, n_jobs=-1)
mcad.obsm[f'5kCG_u{npc}_tsne'] = mcad.obsm['X_tsne'].copy()


In [ ]:
sce.pp.harmony_integrate(mcad, 'Donor', basis=obsm_key, adjusted_basis=f'{obsm_key}_harmony', max_iter_harmony=30, random_state=0)


In [ ]:
tsne(mcad, obsm=f'{obsm_key}_harmony', metric='euclidean', exaggeration=-1, perplexity=50, n_jobs=-1)
mcad.obsm[f'5kCG_u{npc}hm_tsne'] = mcad.obsm['X_tsne'].copy()


In [ ]:
sc.pp.neighbors(mcad, use_rep=f'{obsm_key}_harmony', n_neighbors=25)
sc.tl.leiden(mcad, resolution=1.0, key_added=f'5kCG_u{npc}hm_leiden')

In [ ]:
ds = 4
coord_base = 'tsne'

fig, axes = plt.subplots(2, 2, figsize=(10, 8), dpi=300, constrained_layout=True)

for i,mode in enumerate(['', 'hm']):
    mcad.obsm[f'X_{coord_base}'] = mcad.obsm[f'5kCG_u{npc}{mode}_tsne'].copy()
    dump_embedding(mcad, coord_base)
    tmp = mcad.obs.copy()
    ax = axes[0,i]
    _ = categorical_scatter(data=tmp,
                            ax=ax,
                            coord_base=coord_base,
                            hue='Donor',
                            text_anno='Donor',
                            labelsize=8,
                            s=ds,
                            palette='tab10',
                            scatter_kws={'rasterized':True},
                            # legend_kws={'ncol':1},
                            show_legend=True
                            )
    ax = axes[1,i]
    _ = categorical_scatter(data=tmp,
                            ax=ax,
                            coord_base=coord_base,
                            hue=f'5kCG_u{npc}hm_leiden',
                            text_anno=f'5kCG_u{npc}hm_leiden',
                            labelsize=8,
                            s=ds,
                            palette='tab20',
                            scatter_kws={'rasterized':True},
                            # legend_kws={'ncol':1},
                            # show_legend=True
                            )


In [ ]:
mcad = anndata.AnnData(obs=mcad.obs, obsm=mcad.obsm, obsp=mcad.obsp)
mcad.write_h5ad(f'{outdir}5kCG_embed.h5ad')


In [ ]:
meta = pd.read_csv(f'{indir}clustering/merged/5kCG100k3C_summary.csv.gz', header=0, index_col=0)
meta[['Donor', 'Tissue', 'cluster', 'subtype', 'majortype']] = meta[['Donor', 'Tissue', 'cluster', 'subtype', 'majortype']].astype(str)
meta = meta.loc[meta['Tissue']=='ST']
meta

In [ ]:
mcds = MCDS.open(f'{m3c_dir}*ST*.mcds', use_obs=meta.index, var_dim=var_dim)
mcds = mcds.remove_chromosome(exclude_chromosome=chrom_to_remove, var_dim=var_dim)
mcds

In [ ]:
mcad = mcds.get_score_adata(mc_type='CGN', quant_type='hypo')
binarize_matrix(mcad, cutoff=0.95)


In [ ]:
black_list_path = f'{REF_ROOT}/blacklist/hg38-blacklist.v2.bed.gz'
filter_regions(mcad, n_cell=5)
remove_black_list_region(mcad, black_list_path=black_list_path)


In [ ]:
mcad.write_h5ad(f'{indir}clustering/tissue/L1/ST/5kCG.h5ad')

In [ ]:
mcad = anndata.read_h5ad(f'{indir}clustering/tissue/L1/ST/5kCG.h5ad')

In [ ]:
L1_meta = pd.read_csv(f'{indir}L1color.tsv', sep='\t', header=0, index_col=0)
L1_meta = L1_meta.drop(['c35','c36'], axis=0)
L1_color = L1_meta['color'].to_dict()
L1_annot = L1_meta['L1_abbr'].to_dict()
L2_meta = pd.read_csv(f'{indir}subtype_meta.tsv', sep='\t', header=0, index_col=0)
L2_annot = L2_meta['celltype_L2_both_abbr'].to_dict()

In [ ]:
mcad.obs = meta.loc[mcad.obs.index].copy()
mcad.obs['L2_annot'] = mcad.obs['subtype'].map(L2_annot).astype(str)
mcad.obs['L1_annot'] = mcad.obs['majortype'].map(L1_annot).astype(str)


In [ ]:
adata = anndata.read_h5ad(f'{outdir}5kCG.h5ad')


In [ ]:
tmp = anndata.read_h5ad(f'{outdir}5kCG_embed.h5ad')
adata.obs = tmp.obs.copy()

In [ ]:
selb = mcad.var.index.intersection(adata.var.index)
mcad = mcad[:, selb].copy()
adata = adata[:, selb].copy()
print(mcad.shape, adata.shape)

In [ ]:
mcad.obs['study'] = 'm3C'
adata.obs['study'] = 'mCT'


In [ ]:
adata_merge = anndata.concat([mcad, adata], axis=0, join='outer')
adata_merge

In [ ]:
model = LSI(scale_factor=10000,
            n_components=50,
            algorithm='arpack',
            random_state=0)


In [ ]:
# raw_key = '5kCG_lsi'
# obsm_key = 'X_lsi'
# model.fit_transform(mcad, obsm_name=raw_key)
# npc = significant_pc_test(mcad, p_cutoff=0.05, obsm=raw_key, update=False)
# model.transform(adata, obsm_name=raw_key)

In [ ]:
# mcad = anndata.AnnData(obs=mcad.obs, obsm=mcad.obsm)
# adata = anndata.AnnData(obs=adata.obs, obsm=adata.obsm)
# adata_merge = anndata.concat([mcad, adata], axis=0, join='outer')
# adata_merge

In [ ]:
raw_key = '5kCG_lsi'
obsm_key = 'X_lsi'
model.fit_transform(adata_merge, obsm_name=raw_key)
npc = significant_pc_test(adata_merge, p_cutoff=0.05, obsm=raw_key, update=False)


In [ ]:
npc = 50
adata_merge.obsm[obsm_key] = normalize(adata_merge.obsm[raw_key][:, :npc], axis=1)
sce.pp.harmony_integrate(adata_merge, 'study', basis=obsm_key, adjusted_basis=f'{obsm_key}_harmony', max_iter_harmony=30, random_state=0)
tsne(adata_merge, obsm=f'{obsm_key}_harmony', metric='euclidean', exaggeration=-1, perplexity=50, n_jobs=-1)
adata_merge.obsm[f'5kCG_u{npc}hm_tsne'] = adata_merge.obsm['X_tsne'].copy()

# tsne(adata_merge, obsm=obsm_key, metric='euclidean', exaggeration=-1, perplexity=50, n_jobs=-1)
# adata_merge.obsm[f'5kCG_u{npc}_tsne'] = adata_merge.obsm['X_tsne'].copy()


In [ ]:
ds = 4
coord_base = 'tsne'
adata_merge.obsm[f'X_{coord_base}'] = adata_merge.obsm[f'5kCG_u{npc}hm_tsne'].copy()
dump_embedding(adata_merge, coord_base)

fig, axes = plt.subplots(2, 2, figsize=(10, 8), dpi=300, constrained_layout=True)

tmp = adata_merge.obs.loc[adata_merge.obs['study']=='mCT'].copy()
ax = axes[0,0]
ax.scatter(adata_merge.obsm[f'X_{coord_base}'][:,0], adata_merge.obsm[f'X_{coord_base}'][:,1], s=1, c='lightgrey', alpha=0.5, rasterized=True)
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='Donor',
                        text_anno='Donor',
                        labelsize=8,
                        s=ds,
                        palette='tab10',
                        scatter_kws={'rasterized':True},
                        # legend_kws={'ncol':1},
                        show_legend=True
                        )
ax = axes[0,1]
ax.scatter(adata_merge.obsm[f'X_{coord_base}'][:,0], adata_merge.obsm[f'X_{coord_base}'][:,1], s=1, c='lightgrey', alpha=0.5, rasterized=True)
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue=f'5kCG_u17hm_leiden',
                        text_anno=f'5kCG_u17hm_leiden',
                        labelsize=8,
                        s=ds,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        # legend_kws={'ncol':1},
                        # show_legend=True
                        )

tmp = adata_merge.obs.loc[adata_merge.obs['study']=='m3C'].copy()
ax = axes[1,0]
ax.scatter(adata_merge.obsm[f'X_{coord_base}'][:,0], adata_merge.obsm[f'X_{coord_base}'][:,1], s=1, c='lightgrey', alpha=0.5, rasterized=True)
count = tmp['L1_annot'].value_counts()
tmp = tmp.loc[tmp['L1_annot'].isin(count.index[count>=25])].copy()
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue=f'L1_annot',
                        text_anno=f'L1_annot',
                        labelsize=8,
                        s=ds,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        # legend_kws={'ncol':1},
                        # show_legend=True
                        )

ax = axes[1,1]
ax.scatter(adata_merge.obsm[f'X_{coord_base}'][:,0], adata_merge.obsm[f'X_{coord_base}'][:,1], s=1, c='lightgrey', alpha=0.5, rasterized=True)
count = tmp['L2_annot'].value_counts()
tmp = tmp.loc[tmp['L2_annot'].isin(count.index[count>=25])].copy()
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue=f'L2_annot',
                        text_anno=f'L2_annot',
                        labelsize=8,
                        s=ds,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        # legend_kws={'ncol':1},
                        # show_legend=True
                        )




In [ ]:
ds = 4
coord_base = 'tsne'
adata_merge.obsm[f'X_{coord_base}'] = adata_merge.obsm[f'5kCG_u{npc}hm_tsne'].copy()
dump_embedding(adata_merge, coord_base)

fig, axes = plt.subplots(2, 2, figsize=(10, 8), dpi=300, constrained_layout=True)

tmp = adata_merge.obs.loc[adata_merge.obs['study']=='mCT'].copy()
ax = axes[0,0]
ax.scatter(adata_merge.obsm[f'X_{coord_base}'][:,0], adata_merge.obsm[f'X_{coord_base}'][:,1], s=1, c='lightgrey', alpha=0.5, rasterized=True)
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='Donor',
                        text_anno='Donor',
                        labelsize=8,
                        s=ds,
                        palette='tab10',
                        scatter_kws={'rasterized':True},
                        # legend_kws={'ncol':1},
                        show_legend=True
                        )
ax = axes[0,1]
ax.scatter(adata_merge.obsm[f'X_{coord_base}'][:,0], adata_merge.obsm[f'X_{coord_base}'][:,1], s=1, c='lightgrey', alpha=0.5, rasterized=True)
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue=f'5kCG_u17hm_leiden',
                        text_anno=f'5kCG_u17hm_leiden',
                        labelsize=8,
                        s=ds,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        # legend_kws={'ncol':1},
                        # show_legend=True
                        )

tmp = adata_merge.obs.loc[adata_merge.obs['study']=='m3C'].copy()
ax = axes[1,0]
ax.scatter(adata_merge.obsm[f'X_{coord_base}'][:,0], adata_merge.obsm[f'X_{coord_base}'][:,1], s=1, c='lightgrey', alpha=0.5, rasterized=True)
count = tmp['L1_annot'].value_counts()
tmp = tmp.loc[tmp['L1_annot'].isin(count.index[count>=25])].copy()
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue=f'L1_annot',
                        text_anno=f'L1_annot',
                        labelsize=8,
                        s=ds,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        # legend_kws={'ncol':1},
                        # show_legend=True
                        )

ax = axes[1,1]
ax.scatter(adata_merge.obsm[f'X_{coord_base}'][:,0], adata_merge.obsm[f'X_{coord_base}'][:,1], s=1, c='lightgrey', alpha=0.5, rasterized=True)
count = tmp['L2_annot'].value_counts()
tmp = tmp.loc[tmp['L2_annot'].isin(count.index[count>=25])].copy()
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue=f'L2_annot',
                        text_anno=f'L2_annot',
                        labelsize=8,
                        s=ds,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        # legend_kws={'ncol':1},
                        # show_legend=True
                        )




In [ ]:
cluster_map = {'Epi Gas':[0,1,2,3,4,5,14], 
               'Fibro 1':[6,8,12,15],
               'Fibro 2':[10],
               'Endo Vsc': [11],
               'Myeloid+Mast':[7], 
               'Hema T': [9],
               'Hema B': [13],
               }
adata.obs['celltype'] = adata.obs[f'5kCG_u17hm_leiden'].map(lambda x: next((k for k,v in cluster_map.items() if int(x) in v), 'Unknown')).astype(str)
adata.obs['celltype'].value_counts()

In [ ]:
mcad = anndata.read_h5ad(f'{indir}clustering/tissue/L1/ST/5kCG.h5ad')

In [ ]:
L1_meta = pd.read_csv(f'{indir}L1color.tsv', sep='\t', header=0, index_col=0)
L1_meta = L1_meta.drop(['c35','c36'], axis=0)
L1_color = L1_meta['color'].to_dict()
L1_annot = L1_meta['L1_abbr'].to_dict()
L2_meta = pd.read_csv(f'{indir}subtype_meta.tsv', sep='\t', header=0, index_col=0)
L2_annot = L2_meta['celltype_L2_both_abbr'].to_dict()

In [ ]:
meta = pd.read_csv(f'{indir}clustering/merged/5kCG100k3C_summary.csv.gz', header=0, index_col=0)
meta[['Donor', 'Tissue', 'cluster', 'subtype', 'majortype']] = meta[['Donor', 'Tissue', 'cluster', 'subtype', 'majortype']].astype(str)
meta = meta.loc[meta['Tissue']=='ST']
meta

In [ ]:
mcad.obs = meta.loc[mcad.obs.index].copy()
mcad.obs['L2_annot'] = mcad.obs['subtype'].map(L2_annot).astype(str)
mcad.obs['L1_annot'] = mcad.obs['majortype'].map(L1_annot).astype(str)


In [ ]:
selc = (mcad.obs['L1_annot']=='Epi Gas')
mcad = mcad[selc].copy()


In [ ]:
model = LSI(scale_factor=10000,
            n_components=100,
            algorithm='arpack',
            random_state=0)

raw_key = '5kCG_lsi'
obsm_key = 'X_lsi'
model.fit_transform(mcad, obsm_name=raw_key)
npc = significant_pc_test(mcad, p_cutoff=0.05, obsm=raw_key, update=False)


In [ ]:
mcad.obsm[obsm_key] = normalize(mcad.obsm[raw_key][:, :npc], axis=1)
tsne(mcad, obsm=obsm_key, metric='euclidean', exaggeration=-1, perplexity=50, n_jobs=-1)
mcad.obsm[f'5kCG_u{npc}_tsne'] = mcad.obsm['X_tsne'].copy()


In [ ]:
ds = 4
coord_base = 'tsne'

fig, axes = plt.subplots(1, 2, figsize=(10, 4), dpi=300, constrained_layout=True)

mcad.obsm[f'X_{coord_base}'] = mcad.obsm[f'5kCG_u{npc}_tsne'].copy()
dump_embedding(mcad, coord_base)
tmp = mcad.obs.copy()
ax = axes[0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='Donor',
                        text_anno='Donor',
                        labelsize=8,
                        s=ds,
                        palette='tab10',
                        scatter_kws={'rasterized':True},
                        # legend_kws={'ncol':1},
                        show_legend=True
                        )
ax = axes[1]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue=f'L2_annot',
                        text_anno=f'L2_annot',
                        labelsize=8,
                        s=ds,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        # legend_kws={'ncol':1},
                        show_legend=True
                        )


In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA

label = mcad.obs['L2_annot']
n_class = len(label.unique())
model = LDA(n_components=n_class-1)
mcad.obsm['X_lda'] = model.fit_transform(mcad.obsm[obsm_key], label)


In [ ]:
tsne(mcad, obsm='X_lda', metric='euclidean', exaggeration=-1, perplexity=50, n_jobs=-1)
mcad.obsm[f'5kCG_u{npc}_lda_tsne'] = mcad.obsm['X_tsne'].copy()


In [ ]:
ds = 4
coord_base = 'lda'

fig, axes = plt.subplots(1, 2, figsize=(10, 4), dpi=300, constrained_layout=True)

# mcad.obsm[f'X_{coord_base}'] = mcad.obsm[f'X_lda'].copy()
dump_embedding(mcad, coord_base)
tmp = mcad.obs.copy()
ax = axes[0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='Donor',
                        text_anno='Donor',
                        labelsize=8,
                        s=ds,
                        palette='tab10',
                        scatter_kws={'rasterized':True},
                        # legend_kws={'ncol':1},
                        show_legend=True
                        )
ax = axes[1]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue=f'L2_annot',
                        text_anno=f'L2_annot',
                        labelsize=8,
                        s=ds,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        # legend_kws={'ncol':1},
                        show_legend=True
                        )


In [ ]:
ds = 4
coord_base = 'tsne'

fig, axes = plt.subplots(1, 2, figsize=(10, 4), dpi=300, constrained_layout=True)

mcad.obsm[f'X_{coord_base}'] = mcad.obsm[f'5kCG_u{npc}_lda_tsne'].copy()
dump_embedding(mcad, coord_base)
tmp = mcad.obs.copy()
ax = axes[0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='Donor',
                        text_anno='Donor',
                        labelsize=8,
                        s=ds,
                        palette='tab10',
                        scatter_kws={'rasterized':True},
                        # legend_kws={'ncol':1},
                        show_legend=True
                        )
ax = axes[1]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue=f'L2_annot',
                        text_anno=f'L2_annot',
                        labelsize=8,
                        s=ds,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        # legend_kws={'ncol':1},
                        show_legend=True
                        )


In [ ]:
mcad.obs[['Donor', 'L2_annot']].value_counts().unstack().fillna(0)

In [ ]:
from ALLCools.clustering.lsi import tf_idf
# from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier

scale_factor = 100000
n_components = 100
algorithm = "arpack"
random_state = 0
data = mcad.X.toarray()
tf, idf = tf_idf(data, scale_factor)
# n_rows, n_cols = tf.shape
# label = mcad.obs['L2_annot']
# n_class = len(label.unique())
y = mcad.obs["L2_annot"].values
donors = mcad.obs["Donor"].values

le = LabelEncoder()
y_enc = le.fit_transform(y)
n_classes = len(le.classes_)


In [ ]:
groups = (mcad.obs["L2_annot"].astype(str) + '+' + mcad.obs["Donor"].astype(str)).values

In [ ]:
from sklearn.ensemble import RandomForestClassifier

X_lr = np.zeros((tf.shape[0], n_classes), dtype=float)
# for donor in np.unique(donors):
#     train_idx = donors != donor
#     test_idx  = donors == donor

#     X_train, X_test = tf[train_idx], tf[test_idx]
#     y_train = y_enc[train_idx]

#     clf = LogisticRegression(
#         penalty="l2",
#         solver="lbfgs",
#         multi_class="multinomial",
#         class_weight="balanced",
#         random_state=0,
#         n_jobs=-1
#     )

#     clf.fit(X_train, y_train)

#     # predict probabilities for held-out donor
#     X_lr[test_idx] = clf.predict_proba(X_test)
skf = StratifiedKFold(
    n_splits=2,
    shuffle=True,
    random_state=0
)

for train_idx, test_idx in skf.split(tf, groups):

    # clf = LogisticRegression(
    #     penalty="l2",
    #     solver="lbfgs",
    #     multi_class="multinomial",
    #     class_weight="balanced",
    #     max_iter=1000,
    #     n_jobs=-1
    # )
    clf = RandomForestClassifier(class_weight="balanced")

    clf.fit(tf[train_idx], y_enc[train_idx])

    # out-of-fold predictions
    X_lr[test_idx] = clf.predict_proba(tf[test_idx])

mcad.obsm["X_rf"] = X_lr
    
# model = LDA(n_components=n_class-1)
# tf_reduce = model.fit_transform(mcad.obsm[], label)
# mcad.obsm['X_lda'] = tf_reduce


In [ ]:
mcad.obs['L2_pred'] = le.classes_[np.argmax(mcad.obsm["X_rf"], axis=1)]

In [ ]:
(mcad.obs['L2_pred']==mcad.obs['L2_annot']).sum()

In [ ]:
tsne(mcad, obsm='X_rf', metric='euclidean', exaggeration=-1, perplexity=50, n_jobs=-1)
mcad.obsm[f'rf_tsne'] = mcad.obsm['X_tsne'].copy()


In [ ]:
from sklearn.neighbors import KNeighborsClassifier

pred = np.zeros(tf.shape[0], dtype=int)
for train_idx, test_idx in skf.split(tf, groups):

    # clf = LogisticRegression(
    #     penalty="l2",
    #     solver="lbfgs",
    #     multi_class="multinomial",
    #     class_weight="balanced",
    #     max_iter=1000,
    #     n_jobs=-1
    # )
    clf = KNeighborsClassifier(n_neighbors=15)

    clf.fit(mcad.obsm["X_lr"][train_idx], y_enc[train_idx])

    # out-of-fold predictions
    pred[test_idx] = clf.predict(mcad.obsm["X_lr"][test_idx])

mcad.obs["L2_pred"] = le.classes_[pred]


In [ ]:
mcad.obs["L2_pred"] = le.classes_[pred]
(mcad.obs['L2_pred']==mcad.obs['L2_annot']).sum()

In [ ]:
mcad.write_h5ad(f'{outdir}Epi-Gas_5kCG.h5ad')


In [ ]:
ds = 4
coord_base = 'tsne'

fig, axes = plt.subplots(1, 2, figsize=(10, 4), dpi=300, constrained_layout=True)

mcad.obsm[f'X_{coord_base}'] = mcad.obsm[f'lr_tsne'].copy()
dump_embedding(mcad, coord_base)
tmp = mcad.obs.copy()
ax = axes[0]
ax.set_title('Pred')
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='L2_pred',
                        text_anno='L2_pred',
                        labelsize=8,
                        s=ds,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        # legend_kws={'ncol':1},
                        show_legend=True
                        )
ax = axes[1]
ax.set_title('True')
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue=f'L2_annot',
                        text_anno=f'L2_annot',
                        labelsize=8,
                        s=ds,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        # legend_kws={'ncol':1},
                        show_legend=True
                        )


In [ ]:
ds = 4
coord_base = 'tsne'

fig, axes = plt.subplots(2, 2, figsize=(10, 8), dpi=300, constrained_layout=True)

mcad.obsm[f'X_{coord_base}'] = mcad.obsm[f'lr_tsne'].copy()
dump_embedding(mcad, coord_base)
tmp = mcad.obs.copy()
tmp['score'] = mcad.obsm["X_lr"].max(axis=1)
ax = axes[0, 0]
ax.set_title('Pred')
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='L2_pred',
                        text_anno='L2_pred',
                        labelsize=8,
                        s=ds,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        # legend_kws={'ncol':1},
                        show_legend=True
                        )
ax = axes[0, 1]
ax.set_title('True')
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue=f'L2_annot',
                        text_anno=f'L2_annot',
                        labelsize=8,
                        s=ds,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        # legend_kws={'ncol':1},
                        show_legend=True
                        )
ax = axes[1, 0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='Donor',
                        text_anno='Donor',
                        labelsize=8,
                        s=ds,
                        palette='tab10',
                        scatter_kws={'rasterized':True},
                        # legend_kws={'ncol':1},
                        show_legend=True
                        )
ax = axes[1, 1]
_ = continuous_scatter(data=tmp,
                       ax=ax,
                       coord_base=coord_base,
                       hue='score',
                       labelsize=8,
                       s=ds,
                       cmap='vlag',
                       scatter_kws={'rasterized':True},
                  )


In [ ]:
ds = 4
coord_base = 'tsne'

fig, axes = plt.subplots(2, 2, figsize=(10, 8), dpi=300, constrained_layout=True)

mcad.obsm[f'X_{coord_base}'] = mcad.obsm[f'lr_tsne'].copy()
dump_embedding(mcad, coord_base)
tmp = mcad.obs.copy()
tmp['score'] = mcad.obsm["X_lr"].max(axis=1)
ax = axes[0, 0]
ax.set_title('Pred')
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='L2_pred',
                        text_anno='L2_pred',
                        labelsize=8,
                        s=ds,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        # legend_kws={'ncol':1},
                        show_legend=True
                        )
ax = axes[0, 1]
ax.set_title('True')
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue=f'L2_annot',
                        text_anno=f'L2_annot',
                        labelsize=8,
                        s=ds,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        # legend_kws={'ncol':1},
                        show_legend=True
                        )
ax = axes[1, 0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='Donor',
                        text_anno='Donor',
                        labelsize=8,
                        s=ds,
                        palette='tab10',
                        scatter_kws={'rasterized':True},
                        # legend_kws={'ncol':1},
                        show_legend=True
                        )
ax = axes[1, 1]
_ = continuous_scatter(data=tmp,
                       ax=ax,
                       coord_base=coord_base,
                       hue='score',
                       labelsize=8,
                       s=ds,
                       cmap='vlag',
                       scatter_kws={'rasterized':True},
                  )


In [ ]:
ds = 4
coord_base = 'tsne'

fig, axes = plt.subplots(2, 2, figsize=(10, 8), dpi=300, constrained_layout=True)

mcad.obsm[f'X_{coord_base}'] = mcad.obsm[f'lr_tsne'].copy()
dump_embedding(mcad, coord_base)
tmp = mcad.obs.copy()
tmp['score'] = mcad.obsm["X_lr"].max(axis=1)
ax = axes[0, 0]
ax.set_title('Pred')
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='L2_pred',
                        text_anno='L2_pred',
                        labelsize=8,
                        s=ds,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        # legend_kws={'ncol':1},
                        show_legend=True
                        )
ax = axes[0, 1]
ax.set_title('True')
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue=f'L2_annot',
                        text_anno=f'L2_annot',
                        labelsize=8,
                        s=ds,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        # legend_kws={'ncol':1},
                        show_legend=True
                        )
ax = axes[1, 0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='Donor',
                        text_anno='Donor',
                        labelsize=8,
                        s=ds,
                        palette='tab10',
                        scatter_kws={'rasterized':True},
                        # legend_kws={'ncol':1},
                        show_legend=True
                        )
ax = axes[1, 1]
_ = continuous_scatter(data=tmp,
                       ax=ax,
                       coord_base=coord_base,
                       hue='score',
                       labelsize=8,
                       s=ds,
                       cmap='vlag',
                       scatter_kws={'rasterized':True},
                  )
